In [1]:
from __future__ import annotations

from pathlib import Path
from typing import List

import pandas as pd


AS_OF_DATE = pd.Timestamp("2026-03-17")


def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise FileNotFoundError("Could not locate project root containing pyproject.toml")


def month_range(start_month: pd.Timestamp, end_month: pd.Timestamp) -> pd.DatetimeIndex:
    return pd.date_range(start=start_month, end=end_month, freq="MS")


def normalize_end_date(end_date: str) -> pd.Timestamp:
    if pd.isna(end_date) or str(end_date).strip().lower() == "ongoing":
        return AS_OF_DATE
    return pd.to_datetime(end_date)


def canonical_event_type(category: str) -> str:
    return str(category).strip().lower().replace(" ", "_")


def assign_event_scope(row: pd.Series) -> str:
    event_id = row["Event_ID"]

    trade_policy_ids = {"ECO_001", "ECO_002", "ECO_003", "ECO_004"}
    systemic_ids = {"EVT_001", "EVT_003"}

    if event_id in trade_policy_ids:
        return "trade_policy_regulatory"
    if event_id in systemic_ids:
        return "systemic_macro_disruption"
    return "chokepoint_disruption"


def source_class_for_scope(event_scope: str) -> str:
    source_map = {
        "chokepoint_disruption": "maritime_disruption",
        "systemic_macro_disruption": "macro_systemic",
        "trade_policy_regulatory": "policy_regulatory",
    }
    return source_map[event_scope]


def standardize_chokepoint_name(name: str) -> str:
    raw = str(name).strip()
    rename_map = {
        "Bab el-Mandeb": "Bab el-Mandeb Strait",
    }
    return rename_map.get(raw, raw)


def build_event_geography_links() -> pd.DataFrame:
    # Two-layer geography design:
    # 1) core_chokepoint for canonical PortWatch chokepoints
    # 2) maritime_region for non-core geographies
    links = [
        # Selected systemic spillover context for core story
        {"event_id": "EVT_001", "geo_name": "Suez Canal", "geo_layer": "core_chokepoint", "link_role": "systemic_spillover", "phase_rule": "all"},
        {"event_id": "EVT_001", "geo_name": "Bab el-Mandeb Strait", "geo_layer": "core_chokepoint", "link_role": "systemic_spillover", "phase_rule": "all"},
        {"event_id": "EVT_001", "geo_name": "Panama Canal", "geo_layer": "core_chokepoint", "link_role": "systemic_spillover", "phase_rule": "all"},
        {"event_id": "EVT_001", "geo_name": "Strait of Hormuz", "geo_layer": "core_chokepoint", "link_role": "systemic_spillover", "phase_rule": "all"},
        {"event_id": "EVT_001", "geo_name": "Cape of Good Hope", "geo_layer": "core_chokepoint", "link_role": "systemic_spillover", "phase_rule": "all"},
        {"event_id": "EVT_003", "geo_name": "Strait of Hormuz", "geo_layer": "core_chokepoint", "link_role": "systemic_spillover", "phase_rule": "all"},
        {"event_id": "EVT_003", "geo_name": "Black Sea", "geo_layer": "maritime_region", "link_role": "primary_disruption", "phase_rule": "all"},
        {"event_id": "EVT_003", "geo_name": "Turkish Straits", "geo_layer": "maritime_region", "link_role": "primary_disruption", "phase_rule": "all"},

        # Core chokepoint disruptions
        {"event_id": "EVT_002", "geo_name": "Suez Canal", "geo_layer": "core_chokepoint", "link_role": "primary_disruption", "phase_rule": "all"},
        {"event_id": "EVT_002", "geo_name": "Cape of Good Hope", "geo_layer": "core_chokepoint", "link_role": "reroute_spillover", "phase_rule": "lag_only"},
        {"event_id": "EVT_004", "geo_name": "Panama Canal", "geo_layer": "core_chokepoint", "link_role": "primary_disruption", "phase_rule": "all"},
        {"event_id": "EVT_005", "geo_name": "Bab el-Mandeb Strait", "geo_layer": "core_chokepoint", "link_role": "primary_disruption", "phase_rule": "all"},
        {"event_id": "EVT_005", "geo_name": "Suez Canal", "geo_layer": "core_chokepoint", "link_role": "secondary_spillover", "phase_rule": "active_lag"},
        {"event_id": "EVT_005", "geo_name": "Cape of Good Hope", "geo_layer": "core_chokepoint", "link_role": "reroute_spillover", "phase_rule": "active_lag"},
        {"event_id": "EVT_006", "geo_name": "Strait of Hormuz", "geo_layer": "core_chokepoint", "link_role": "primary_disruption", "phase_rule": "all"},
        {"event_id": "EVT_007", "geo_name": "Strait of Hormuz", "geo_layer": "core_chokepoint", "link_role": "primary_disruption", "phase_rule": "all"},
        {"event_id": "EVT_008", "geo_name": "Strait of Hormuz", "geo_layer": "core_chokepoint", "link_role": "primary_disruption", "phase_rule": "all"},
        {"event_id": "EVT_008", "geo_name": "Gulf of Oman", "geo_layer": "maritime_region", "link_role": "secondary_spillover", "phase_rule": "all"},
        {"event_id": "PHY_001", "geo_name": "Port of Baltimore", "geo_layer": "maritime_region", "link_role": "primary_disruption", "phase_rule": "all"},
        {"event_id": "PHY_001", "geo_name": "US East Coast", "geo_layer": "maritime_region", "link_role": "secondary_spillover", "phase_rule": "all"},
    ]
    links_df = pd.DataFrame(links)
    links_df["geo_name"] = links_df["geo_name"].apply(standardize_chokepoint_name)
    return links_df


def is_phase_allowed(event_phase: str, phase_rule: str) -> bool:
    if phase_rule == "all":
        return True
    if phase_rule == "active_lag":
        return event_phase in {"active", "lag"}
    if phase_rule == "lag_only":
        return event_phase == "lag"
    return False


def phase_rows_for_event(row: pd.Series) -> List[dict]:
    start_month = pd.to_datetime(row["Start_Date"]).to_period("M").to_timestamp()
    end_month = normalize_end_date(row["End_Date"]).to_period("M").to_timestamp()

    lead_months = int(row["Lead_Months"])
    lag_months = int(row["Lag_Months"])

    phase_defs = []

    if lead_months < 0:
        lead_start = start_month + pd.offsets.MonthBegin(lead_months)
        lead_end = start_month - pd.offsets.MonthBegin(1)
        for m in month_range(lead_start, lead_end):
            phase_defs.append((m, "lead"))

    for m in month_range(start_month, end_month):
        phase_defs.append((m, "active"))

    if lag_months > 0:
        lag_start = end_month + pd.offsets.MonthBegin(1)
        lag_end = end_month + pd.offsets.MonthBegin(lag_months)
        for m in month_range(lag_start, lag_end):
            phase_defs.append((m, "lag"))

    phase_mult = {"lead": 0.70, "active": 1.00, "lag": 0.50}

    out = []
    for month_ts, phase in phase_defs:
        out.append(
            {
                "event_id": row["Event_ID"],
                "event_name": row["Event_Name"],
                "year_month": month_ts.strftime("%Y-%m"),
                "event_phase": phase,
                "event_active_flag": int(phase == "active"),
                "lead_flag": int(phase == "lead"),
                "lag_flag": int(phase == "lag"),
                "severity_weight": round(float(row["base_severity"]) * phase_mult[phase], 2),
                "global_event_flag": int(row["event_scope"] == "systemic_macro_disruption"),
                "event_type": row["event_type"],
                "event_scope": row["event_scope"],
            }
        )

    return out


def write_partitioned_parquet(df: pd.DataFrame, out_dir: Path) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)
    for ym, chunk in df.groupby("year_month", sort=True):
        part_dir = out_dir / f"year_month={ym}"
        part_dir.mkdir(parents=True, exist_ok=True)
        chunk.to_parquet(part_dir / "part-000.parquet", index=False)


PROJECT_ROOT = find_project_root(Path.cwd())
BRONZE_EVENTS = PROJECT_ROOT / "data" / "bronze" / "events.csv"
SILVER_EVENTS_DIR = PROJECT_ROOT / "data" / "silver" / "events"
CORE_DIR = SILVER_EVENTS_DIR / "bridge_event_month_chokepoint_core"
REGION_DIR = SILVER_EVENTS_DIR / "bridge_event_month_maritime_region"

SILVER_EVENTS_DIR.mkdir(parents=True, exist_ok=True)


events = pd.read_csv(BRONZE_EVENTS)

events["event_type"] = events["Event_Category"].apply(canonical_event_type)
events["event_scope"] = events.apply(assign_event_scope, axis=1)

events["base_severity"] = events["Event_Category"].map(
    {
        "Systemic Global": 1.00,
        "Geopolitical": 0.85,
        "Physical Chokepoint": 0.75,
        "Economic Policy": 0.60,
    }
).fillna(0.60)

events["source_class"] = events["event_scope"].apply(source_class_for_scope)

events["Start_Date"] = pd.to_datetime(events["Start_Date"]).dt.date
events["End_Date"] = events["End_Date"].apply(
    lambda x: normalize_end_date(x).date() if str(x).strip().lower() == "ongoing" else pd.to_datetime(x).date()
)

# Build dim_event.
dim_event = events.rename(
    columns={
        "Event_ID": "event_id",
        "Event_Name": "event_name",
        "Start_Date": "start_date",
        "End_Date": "end_date",
        "Lead_Months": "lead_months",
        "Lag_Months": "lag_months",
        "Description_Effects": "description",
    }
)[
    [
        "event_id",
        "event_name",
        "event_type",
        "event_scope",
        "start_date",
        "end_date",
        "lead_months",
        "lag_months",
        "base_severity",
        "description",
        "source_class",
    ]
].sort_values("event_id").reset_index(drop=True)

# Bridge eligibility excludes policy events from chokepoint bridge layers.
bridge_eligible_ids = dim_event[
    (dim_event["event_scope"] == "chokepoint_disruption")
    | (dim_event["event_id"].isin(["EVT_001", "EVT_003"]))
]["event_id"].tolist()

events_for_bridge = events[events["Event_ID"].isin(bridge_eligible_ids)].copy()

phase_records = []
for _, event_row in events_for_bridge.iterrows():
    phase_records.extend(phase_rows_for_event(event_row))
bridge_base = pd.DataFrame(phase_records)

links = build_event_geography_links()
bridge_all = bridge_base.merge(links, on="event_id", how="inner")
bridge_all = bridge_all[bridge_all.apply(lambda r: is_phase_allowed(r["event_phase"], r["phase_rule"]), axis=1)]

bridge_all = bridge_all.rename(columns={"geo_name": "chokepoint_name"})
bridge_all["chokepoint_name"] = bridge_all["chokepoint_name"].apply(standardize_chokepoint_name)

bridge_cols = [
    "event_id",
    "event_name",
    "year_month",
    "chokepoint_name",
    "event_phase",
    "event_active_flag",
    "lead_flag",
    "lag_flag",
    "severity_weight",
    "global_event_flag",
    "event_type",
    "event_scope",
    "link_role",
]

bridge_event_month_chokepoint_core = bridge_all[bridge_all["geo_layer"] == "core_chokepoint"][bridge_cols]
bridge_event_month_maritime_region = bridge_all[bridge_all["geo_layer"] == "maritime_region"][bridge_cols]

bridge_event_month_chokepoint_core = bridge_event_month_chokepoint_core.drop_duplicates(
    subset=["event_id", "year_month", "chokepoint_name", "event_phase", "link_role"]
).sort_values(["year_month", "event_id", "chokepoint_name", "event_phase"]).reset_index(drop=True)

bridge_event_month_maritime_region = bridge_event_month_maritime_region.drop_duplicates(
    subset=["event_id", "year_month", "chokepoint_name", "event_phase", "link_role"]
).sort_values(["year_month", "event_id", "chokepoint_name", "event_phase"]).reset_index(drop=True)

write_partitioned_parquet(bridge_event_month_chokepoint_core, CORE_DIR)
write_partitioned_parquet(bridge_event_month_maritime_region, REGION_DIR)

bridge_event_month_chokepoint_core.to_csv(
    SILVER_EVENTS_DIR / "bridge_event_month_chokepoint_core.csv", index=False
)
bridge_event_month_maritime_region.to_csv(
    SILVER_EVENTS_DIR / "bridge_event_month_maritime_region.csv", index=False
)
dim_event.to_csv(SILVER_EVENTS_DIR / "dim_event.csv", index=False)
dim_event.to_parquet(SILVER_EVENTS_DIR / "dim_event.parquet", index=False)

print(f"Core bridge rows: {len(bridge_event_month_chokepoint_core):,}")
print(f"Core bridge events: {bridge_event_month_chokepoint_core['event_id'].nunique():,}")
print(f"Region bridge rows: {len(bridge_event_month_maritime_region):,}")
print(f"Region bridge events: {bridge_event_month_maritime_region['event_id'].nunique():,}")
print(f"Dim events: {len(dim_event):,}")
print(f"Core partition root: {CORE_DIR}")
print(f"Region partition root: {REGION_DIR}")

print("\nCore chokepoint names:")
print(sorted(bridge_event_month_chokepoint_core["chokepoint_name"].unique()))

print("\nEvent scope counts:")
print(dim_event["event_scope"].value_counts())

bridge_event_month_chokepoint_core.head(20)

Core bridge rows: 306
Core bridge events: 8
Region bridge rows: 137
Region bridge events: 3
Dim events: 13
Core partition root: /Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/silver/events/bridge_event_month_chokepoint_core
Region partition root: /Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/silver/events/bridge_event_month_maritime_region

Core chokepoint names:
['Bab el-Mandeb Strait', 'Cape of Good Hope', 'Panama Canal', 'Strait of Hormuz', 'Suez Canal']

Event scope counts:
event_scope
chokepoint_disruption        7
trade_policy_regulatory      4
systemic_macro_disruption    2
Name: count, dtype: int64


,event_id,event_name,year_month,chokepoint_name,event_phase,event_active_flag,lead_flag,lag_flag,severity_weight,global_event_flag,event_type,event_scope,link_role
0,EVT_001,COVID-19 Global Shock,2019-12,Bab el-Mandeb Strait,lead,0,1,0,0.7,1,systemic_global,systemic_macro_disruption,systemic_spillover
1,EVT_001,COVID-19 Global Shock,2019-12,Cape of Good Hope,lead,0,1,0,0.7,1,systemic_global,systemic_macro_disruption,systemic_spillover
2,EVT_001,COVID-19 Global Shock,2019-12,Panama Canal,lead,0,1,0,0.7,1,systemic_global,systemic_macro_disruption,systemic_spillover
3,EVT_001,COVID-19 Global Shock,2019-12,Strait of Hormuz,lead,0,1,0,0.7,1,systemic_global,systemic_macro_disruption,systemic_spillover
4,EVT_001,COVID-19 Global Shock,2019-12,Suez Canal,lead,0,1,0,0.7,1,systemic_global,systemic_macro_disruption,systemic_spillover
5,EVT_001,COVID-19 Global Shock,2020-01,Bab el-Mandeb Strait,active,1,0,0,1.0,1,systemic_global,systemic_macro_disruption,systemic_spillover
6,EVT_001,COVID-19 Global Shock,2020-01,Cape of Good Hope,active,1,0,0,1.0,1,systemic_global,systemic_macro_disruption,systemic_spillover
7,EVT_001,COVID-19 Global Shock,2020-01,Panama Canal,active,1,0,0,1.0,1,systemic_global,systemic_macro_disruption,systemic_spillover
8,EVT_001,COVID-19 Global Shock,2020-01,Strait of Hormuz,active,1,0,0,1.0,1,systemic_global,systemic_macro_disruption,systemic_spillover
9,EVT_001,COVID-19 Global Shock,2020-01,Suez Canal,active,1,0,0,1.0,1,systemic_global,systemic_macro_disruption,systemic_spillover
